#실습 — RAG가 무엇이고 왜 필요한가

**위니브 카페 CS 챗봇 만들기 **



## 오늘 배울 것 

> AI는 모르는 걸 물으면 **그럴듯하게 지어냅니다(거짓말).**
> 그런데 **문서를 같이 보여주면** 정확하게 답합니다. 이게 RAG입니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | AI는 왜 거짓말을 하나? RAG란? |
| 2️⃣ 기본 | AI를 코드로 불러보기 (가장 쉬운 형태) |
| 3️⃣ 응용 | 프롬프트로 AI에게 역할·규칙 주기 |
| 4️⃣ 실습 | 위니브 문서로 "거짓말 → 정확" 직접 확인 |


# 1️⃣ 단계 — 이론 

## AI는 왜 거짓말을 할까?

ChatGPT 같은 AI를 **LLM** 이라고 부릅니다. AI는 인터넷의 수많은
글을 학습했습니다. 그래서 일반 상식은 잘 압니다.

하지만 **위니브 카페의 환불 규정** 같은 건 인터넷에 없습니다.
AI는 이걸 배운 적이 없어요. 그런데 물어보면 어떻게 될까요?

> 🧑 고객: "개봉한 원두 환불돼요?"
> 🤖 AI: "네, 30일 이내면 가능합니다!" ← **그럴듯하지만 틀림**

AI는 "모릅니다"라고 하기보다 **그럴듯한 말을 지어냅니다.**
이걸 **환각(Hallucination)** 이라고 합니다.

> 💡 비유: 회사 규정집을 한 번도 안 본 신입사원에게 "환불 규정이
> 뭐죠?"라고 물으면, 성실한 신입은 "보통 30일쯤?" 하고 **추측으로
> 자신 있게** 답합니다. AI가 딱 이 신입입니다.

## 고객센터에서 이게 왜 큰 문제일까?

챗봇이 "환불 됩니다"라고 잘못 안내하면 → 고객이 믿고 신청 →
거절당하면 → **항의·분쟁**. 회사 신뢰가 떨어집니다.

## 해결책: RAG

신입이 규정을 몰라서 지어낸다면? 답은 간단합니다.

> 🗝️ **답하기 전에 규정집을 펴서, 그걸 보고 답하게 하자.**

이걸 AI에 적용한 게 **RAG (Retrieval-Augmented Generation)** 입니다.
어려운 말이지만 핵심은 하나입니다:

> **AI에게 질문만 주지 말고, 관련 문서도 함께 줘서, 그걸 보고
> 답하게 한다.** (= 시험을 "오픈북"으로 바꾼다)

오늘은 가장 단순한 RAG — **문서를 통째로 같이 주기** — 를
직접 해봅니다.

---



# 2️⃣ 단계 — AI를 코드로 불러보기

**"AI에게 질문하고 답을 받는다."**

### 셀 1 — 필요한 도구 설치

AI를 부르려면 도구(라이브러리)가 필요합니다. 한 번만 설치하면 됩니다.

In [ ]:
!pip install -q langchain langchain-openai langsmith python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 12.9 MB/s eta 0:00:00


### 셀 2 — API 키 넣기

AI(OpenAI)를 쓰려면 열쇠(API 키)가 필요합니다.
- **Colab**: 왼쪽 🔑 모양 버튼 → `OPENAI_API_KEY`, `LANGSMITH_API_KEY` 추가
- **로컬**: 같은 폴더에 `.env` 파일 만들고 키 두 줄 작성

아래 셀은 그 키를 자동으로 읽어옵니다. 

In [ ]:
import os

# Colab이면 보안 비밀에서, 아니면 .env 파일에서 키를 읽어옴
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# LangSmith 추적 켜기 (AI가 한 일이 기록으로 남음 — 나중에 디버깅용)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "winib-cs-lesson1"

# 키가 잘 들어왔는지 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✅ 준비 완료! 다음 셀로 넘어가세요.")
else:
    print("❌ OPENAI_API_KEY가 없습니다. 위 안내대로 키를 넣어주세요.")

✅ 준비 완료! 다음 셀로 넘어가세요.


AI를 쓸 열쇠를 환경에 등록했습니다.
"✅ 준비 완료"가 보이면 성공입니다.

### 셀 3 — AI에게 처음으로 질문하기

이제 AI를 부릅니다. 
1. AI 모델을 준비하고 (`llm`)
2. 질문을 던지고 (`.invoke`)
3. 답을 출력합니다.

In [ ]:
from langchain_openai import ChatOpenAI

# AI 모델 준비 (gpt-4o-mini = 빠르고 저렴한 모델)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# AI에게 질문하고 답 받기
answer = llm.invoke("안녕하세요! 커피를 한 문장으로 소개해줄래요?")

# 답 출력 (.content 안에 답 글자가 들어있음)
print(answer.content)

안녕하세요! 커피는 풍부한 향과 깊은 맛을 지닌 음료로, 전 세계에서 사랑받으며 사람들을 연결하는 특별한 음료입니다.


**이 셀이 한 일**: AI에게 질문하고 답을 받아 출력했습니다.
이게 AI를 코드로 쓰는 가장 기본 형태입니다.

> 💡 `temperature=0` 은 "매번 비슷하고 안정적인 답"을 뜻합니다.
> 고객센터는 일관성이 중요하니 항상 0을 씁니다.

### 셀 4 — AI는 위니브 규정을 모른다 (문제 확인)

이제 위니브 카페 환불 규정을 물어봅니다.
AI는 이 규정을 배운 적이 없습니다. 어떻게 답할까요?

In [ ]:
question = "위니브 카페에서 개봉한 원두를 단순 변심으로 환불할 수 있나요?"

answer = llm.invoke(question)
print(answer.content)

위니브 카페의 환불 정책은 매장이나 온라인 구매에 따라 다를 수 있습니다. 일반적으로 개봉한 원두는 단순 변심으로 환불이 어려운 경우가 많습니다. 하지만 정확한 정보는 위니브 카페의 공식 웹사이트나 고객 서비스에 문의하여 확인하는 것이 가장 좋습니다. 각 매장마다 정책이 다를 수 있으니, 직접 확인해 보시기 바랍니다.


**이 셀이 한 일**: 위니브 규정을 AI에게 물었습니다.

> ⚠️ AI가 "가능하다"는 식으로 답했을 겁니다. 하지만 위니브
> 실제 규정은 **"개봉한 원두는 환불 불가"** 입니다.
> AI가 모르면서 지어낸 것 — 이게 1단계에서 배운 **환각**입니다.

이 문제를 4단계에서 RAG로 해결합니다. 그 전에 3단계에서
"AI에게 역할 주는 법"을 배웁니다.

# 3️⃣ 단계 — AI에게 역할과 규칙 주기 (프롬프트)

셀 3·4에서는 질문만 던졌습니다. 하지만 실제 챗봇은
**"너는 상담원이야. 이렇게 답해."** 라고 역할을 정해줍니다.
이 "지시문"을 **프롬프트(Prompt)** 라고 합니다.

### 셀 5 — 프롬프트로 상담원 역할 주기

프롬프트는 두 부분으로 만듭니다.
- `system` : AI의 역할·규칙 ("너는 위니브 상담원이야")
- `human` : 고객의 질문이 들어갈 자리

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# AI에게 줄 지시문(프롬프트) 만들기
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 위니브 카페의 친절한 고객센터 상담원입니다. 정중한 존댓말로 답하세요."),
    ("human", "{question}"),   # {question} 자리에 실제 질문이 들어감
])

# 프롬프트 → AI → 답(글자만 추출) 을 하나로 연결
chatbot = prompt | llm | StrOutputParser()

# 질문을 넣어 실행 (question 자리에 값을 채워줌)
result = chatbot.invoke({"question": "영업시간이 언제예요?"})
print(result)

안녕하세요, 위니브 카페에 문의해 주셔서 감사합니다. 저희 영업시간은 평일 오전 9시부터 오후 10시까지, 주말 및 공휴일은 오전 10시부터 오후 9시까지입니다. 추가로 궁금하신 사항이 있으시면 언제든지 말씀해 주세요!


**이 셀이 한 일**: AI에게 "위니브 상담원 역할"을 주고 답하게
했습니다. 답투가 상담원처럼 정중해졌죠?

> 💡 `prompt | llm | StrOutputParser()` 의 `|` 기호는
> "이것 다음 저것" 순서로 연결한다는 뜻입니다.
> (프롬프트로 만들고 → AI에 넣고 → 답 글자만 깔끔히 빼기)

### 셀 6 — 역할만 바꿔도 답이 달라진다

`system` 지시문만 바꿔봅니다. 코드 구조는 셀 5와 똑같습니다.

In [ ]:
# 이번엔 "한 문장으로만 짧게" 답하라고 지시
short_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 위니브 카페 상담원입니다. 반드시 한 문장으로만 짧게 답하세요."),
    ("human", "{question}"),
])
short_chatbot = short_prompt | llm | StrOutputParser()

print(short_chatbot.invoke({"question": "회원 가입하면 뭐가 좋아요?"}))

회원 가입 시 다양한 혜택과 이벤트에 참여할 수 있습니다.


**이 셀이 한 일**: 지시문만 "짧게 답해"로 바꿨더니 답도
짧아졌습니다.

> 📌 정리: AI의 행동은 **프롬프트(지시문)로 조종**합니다.
> 코드 구조(`prompt | llm | StrOutputParser()`)는 그대로 두고
> 프롬프트만 바꾸면 됩니다. — 이게 4단계 RAG의 핵심이 됩니다.

# 4️⃣ 단계 — 위니브 문서로 거짓말 고치기 (RAG!)

드디어 오늘의 핵심입니다. 셀 4에서 AI가 환불 규정을 지어냈죠.
이번엔 **규정 문서를 같이 줘서** 정확하게 답하게 합니다.

### 셀 7 — 위니브 환불 규정 문서 준비

위니브 카페의 실제 환불 규정이 아래 변수에 들어 있습니다.
(강사가 만든 진짜 문서입니다)

In [ ]:
refund_policy = "# 위니브 카페 환불·반품·교환 정책\n\n본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.\n\n## 1. 반품·환불 신청 기간\n\n- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능\n- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능\n- 위 기간이 지나면 반품·환불이 제한될 수 있습니다.\n\n## 2. 환불 불가 품목 (중요)\n\n다음 품목은 단순 변심에 의한 반품·환불이 **불가**합니다.\n\n- **개봉한 원두 및 드립백**: 식품 위생상 개봉 시 재판매가 불가능합니다.\n  미개봉 상태에서는 7일 이내 반품 가능합니다.\n- **디저트류(케이크·쿠키·구움과자 등)**: 신선식품으로, 단순 변심 반품이\n  불가합니다. 단, 변질·파손 등 하자가 있는 경우 수령 14일 이내 교환·환불됩니다.\n- **주문 제작 굿즈(각인·커스텀 텀블러 등)**: 고객 요청으로 제작된 상품은\n  단순 변심 반품이 불가합니다.\n- 단, 위 품목도 **상품 하자나 오배송**인 경우에는 환불·교환이 가능합니다.\n\n## 3. 단순 변심 반품 시 비용\n\n- 단순 변심으로 반품할 경우 왕복 배송비(편도 3,000원 × 2 = 6,000원)는\n  고객이 부담합니다.\n- 상품 하자·오배송으로 인한 반품은 배송비를 위니브 카페가 전액 부담합니다.\n\n## 4. 환불 처리 절차 및 기간\n\n1. 마이페이지 > 주문내역에서 반품 신청\n2. 반품 상품 발송 (회수 택배 신청 또는 직접 발송)\n3. 반품 상품 도착 후 상태 확인\n4. 확인 완료 후 **영업일 기준 3일 이내** 환불 처리\n\n- 카드 결제는 카드사 정책에 따라 취소 반영까지 영업일 3~5일이 추가로\n  소요될 수 있습니다.\n- 적립금으로 결제한 부분은 적립금으로 환원됩니다.\n\n## 5. 교환\n\n- 동일 상품의 색상·옵션 변경은 재고가 있는 경우 교환 가능합니다.\n- 하자 상품의 교환 배송비는 위니브 카페가 부담합니다.\n- 단순 변심에 의한 교환 배송비는 고객이 부담합니다.\n- 디저트류는 교환이 어려워 환불로 처리될 수 있습니다.\n\n## 6. 주문 취소\n\n- 상품 발송 **전**에는 마이페이지에서 즉시 취소 가능하며 전액 환불됩니다.\n- 발송 **후**에는 취소가 불가하며, 수령 후 반품 절차로 진행됩니다.\n\n## 7. 환불이 제한되는 경우\n\n- 고객의 사용·훼손으로 상품 가치가 현저히 감소한 경우\n- 개봉 후 상태인 식품(원두·드립백·디저트)의 단순 변심\n- 반품 기간(7일/14일)을 경과한 경우\n- 위 사유라도 상품 자체 하자가 입증되면 환불 대상이 됩니다."
shipping_policy = "# 위니브 카페 배송 정책\n\n## 1. 배송비\n\n| 구분 | 배송비 |\n|------|--------|\n| 기본 배송비 | 3,000원 |\n| 5만원 이상 구매 | 무료 |\n| 제주 지역 | 기본 배송비 + 3,000원 |\n| 도서산간 지역 | 기본 배송비 + 3,000원 |\n\n- 5만원 이상 구매 시 기본 배송비는 무료이나, 제주·도서산간 추가비\n  (3,000원)는 별도로 부과됩니다.\n\n## 2. 배송 기간\n\n- 평일 **오후 2시 이전** 결제 완료 건은 당일 발송됩니다.\n- 오후 2시 이후 결제 건은 다음 영업일에 발송됩니다.\n- 발송 후 도착까지 보통 1~2일 소요됩니다.\n- 제주·도서산간은 2~4일이 소요될 수 있습니다.\n- 주말·공휴일은 발송되지 않습니다.\n\n## 3. 원두 로스팅 일정\n\n- 원두는 신선도를 위해 **주문 후 로스팅(주문 로스팅)** 으로 발송됩니다.\n- 로스팅은 매주 화·금요일에 진행되며, 해당 요일 직전 주문이 함께\n  발송됩니다. 이로 인해 원두 단일 주문은 최대 3~4일이 소요될 수 있습니다.\n- 원두와 일반 상품을 함께 주문하면 로스팅 일정에 맞춰 합포장 발송됩니다.\n\n## 4. 배송 조회\n\n- 마이페이지 > 주문내역에서 송장번호로 실시간 조회가 가능합니다.\n- 발송 완료 시 등록된 휴대폰으로 알림 문자가 발송됩니다.\n\n## 5. 배송 관련 안내\n\n- 택배사는 일반 택배(CJ대한통운)를 이용합니다.\n- **새벽배송·당일배송 서비스는 운영하지 않습니다.**\n- 부재 시 택배사 정책에 따라 경비실 위탁 또는 재배송됩니다.\n- 배송지 오기입으로 인한 반송 시 재발송 배송비는 고객이 부담합니다.\n\n## 6. 해외 배송\n\n- 현재 해외 배송은 지원하지 않습니다. 국내 배송만 가능합니다."
membership = "# 위니브 카페 멤버십·적립 안내\n\n## 1. 회원 등급\n\n회원 등급은 **최근 6개월 누적 구매액**을 기준으로 매월 1일 자동 산정됩니다.\n\n| 등급 | 최근 6개월 누적 구매액 | 적립률 |\n|------|----------------------|--------|\n| 베이직 | 0원 ~ | 1% |\n| 실버 | 10만원 이상 | 2% |\n| 골드 | 30만원 이상 | 3% |\n| VIP | 60만원 이상 | 5% |\n\n- 등급은 매월 1일 재산정되며, 기준 미달 시 하향될 수 있습니다.\n- 신규 가입 시 기본 등급은 베이직입니다.\n\n## 2. 적립금\n\n- 구매 확정 시 등급별 적립률에 따라 적립됩니다.\n- 적립금은 **적립일로부터 1년** 경과 시 자동 소멸됩니다.\n- 적립금은 1,000원 이상부터 사용 가능하며, 1원 단위로 사용할 수 있습니다.\n- 비회원 주문은 적립이 적용되지 않습니다.\n\n## 3. 신규 가입 혜택\n\n- 신규 회원가입 시 3,000원 웰컴 적립금이 즉시 지급됩니다.\n- 웰컴 적립금의 유효기간은 지급일로부터 30일입니다.\n\n## 4. 생일 쿠폰\n\n- 생일이 등록된 회원에게 생일月 1일에 10% 할인 쿠폰이 발급됩니다.\n- 생일 쿠폰은 발급일로부터 그 달 말일까지 사용 가능합니다.\n- 일부 할인 상품에는 중복 적용되지 않습니다.\n\n## 5. 적립금 환불 처리\n\n- 적립금으로 결제한 주문을 환불할 경우, 사용한 적립금은 적립금으로\n  환원됩니다(현금 환불 아님).\n- 환원된 적립금의 유효기간은 원래 적립금의 잔여 기간을 따릅니다.\n\n## 6. 등급 혜택 요약\n\n- 골드 등급 이상: 무료 선물포장 제공\n- VIP 등급: 신상품 우선 구매 및 분기별 시음 원두 증정"

# 환불 규정 앞부분만 살짝 확인
print(refund_policy[:200], "...")

# 위니브 카페 환불·반품·교환 정책

본 정책은 전자상거래법을 따르며, 위니브 카페에서 구매한 모든 상품에 적용됩니다.

## 1. 반품·환불 신청 기간

- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능
- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능
- 위 기간이 지나면 반품·환불이 제한될 수 있습니다.

## 2. 환불 ...


**이 셀이 한 일**: 위니브 규정 문서 3개를 변수에 담았습니다.
오늘은 `refund_policy`(환불 규정)를 주로 씁니다.

### 셀 8 — 문서를 같이 줘서 답하게 하기 (= RAG)

핵심 아이디어: 프롬프트에 **규정 문서를 넣는 자리** `{document}` 를
하나 더 만듭니다. 그리고 "이 규정만 보고 답해"라고 지시합니다.

In [ ]:
# 규정 문서를 함께 주는 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 위니브 카페 상담원입니다. "
     "아래 [규정]만 보고 답하세요. 규정에 없으면 모른다고 하세요.\n\n"
     "[규정]\n{document}"),
    ("human", "{question}"),
])
rag_chatbot = rag_prompt | llm | StrOutputParser()

# 셀 4와 똑같은 질문을, 이번엔 규정 문서와 함께
question = "위니브 카페에서 개봉한 원두를 단순 변심으로 환불할 수 있나요?"
answer = rag_chatbot.invoke({
    "document": refund_policy,   # ← 규정 문서를 통째로 넣음
    "question": question,
})
print(answer)

개봉한 원두는 식품 위생상 재판매가 불가능하므로 단순 변심으로 환불이 불가합니다.


**이 셀이 한 일**: 셀 4와 똑같은 질문인데, 이번엔 **규정 문서를
같이 줬습니다.**

> ✅ 이번엔 "개봉한 원두는 단순변심 환불 **불가**"라고 정확히
> 답했을 겁니다. 셀 4(거짓말)와 비교해 보세요.
> **문서를 같이 준 것만으로 거짓말이 사라졌습니다. 이게 RAG입니다!**

### 셀 9 — 나란히 비교 (문서 없이 vs 있이)

같은 질문을 두 방식으로 답하게 해서 차이를 직접 봅니다.

In [ ]:
q = "케이크(디저트)를 단순 변심으로 반품할 수 있나요?"

# (1) 문서 없는 — 셀 5의 일반 챗봇
print("───[ 문서 없이 ]───")
print(chatbot.invoke({"question": q}))

# (2) 문서 있는 — 셀 8의 RAG 챗봇
print("\n───[ 문서 있이 (RAG) ]───")
print(rag_chatbot.invoke({"document": refund_policy, "question": q}))

───[ 문서 없이 ]───
안녕하세요, 위니브 카페 고객센터입니다. 케이크와 같은 디저트는 단순 변심으로 인한 반품이 어려운 점 양해 부탁드립니다. 제품의 특성상 신선도가 중요한 만큼, 변심으로 인한 반품은 불가합니다. 다른 문의사항이 있으시면 언제든지 말씀해 주세요. 감사합니다!

───[ 문서 있이 (RAG) ]───
케이크(디저트)는 신선식품으로 단순 변심에 의한 반품이 불가합니다. 변질·파손 등 하자가 있는 경우에는 수령 14일 이내에 교환·환불이 가능합니다.


**이 셀이 한 일**: 같은 질문을 문서 없이 / 있이 답하게 했습니다.

> - 문서 없이: 그럴듯하게 지어낼 수 있음
> - 문서 있이: "디저트는 신선식품이라 단순변심 반품 불가" 라고 정확
>
> 이 차이가 고객센터에서는 **분쟁을 막느냐 마느냐**의 차이입니다.

### 셀 10 — 다른 문서로도 해보기 (멤버십)

같은 RAG 챗봇에 **다른 문서**(멤버십 규정)를 넣어봅니다.
코드는 셀 8과 똑같고, `document` 만 바꾸면 됩니다.

In [ ]:
q = "실버 등급이 되려면 얼마를 사야 하고, 적립률은 몇 %인가요?"

answer = rag_chatbot.invoke({
    "document": membership,   # ← 이번엔 멤버십 문서
    "question": q,
})
print(answer)

실버 등급이 되려면 최근 6개월 누적 구매액이 10만원 이상이어야 하며, 적립률은 2%입니다.


**이 셀이 한 일**: 멤버십 문서를 넣으니 등급·적립률을 정확히
답합니다.

> 📌 깨달음: RAG 챗봇은 그대로 두고 **문서만 바꿔 끼우면** 어떤
> 주제든 정확히 답합니다. 문서가 곧 챗봇의 지식입니다.

### 셀 11 — 문제점 발견: 문서가 길면? (다음 시간 예고)

지금은 문서를 **통째로** 넣었습니다. 문서가 아주 길거나
여러 개면 어떻게 될까요? 토큰(AI가 글을 세는 단위)을 봅니다.

In [ ]:
# 규정을 일부러 5배로 길게 만들어 봄
long_doc = refund_policy * 5

# 답을 받되, AI가 글을 얼마나 처리했는지(토큰)도 확인
response = (rag_prompt | llm).invoke({
    "document": long_doc,
    "question": "환불은 며칠 이내에 신청해야 하나요?",
})
print("AI가 읽은 글자량(입력 토큰):", response.usage_metadata["input_tokens"])
print("\n답변:", response.content[:100], "...")

AI가 읽은 글자량(입력 토큰): 3913

답변: 환불 신청 기간은 다음과 같습니다:

- **단순 변심**: 상품 수령일로부터 7일 이내 신청 가능
- **상품 하자·오배송**: 수령일로부터 14일 이내 신청 가능

위 기간이  ...


**이 셀이 한 일**: 문서를 5배로 늘렸더니 AI가 읽어야 할
양(토큰)이 크게 늘었습니다. 토큰이 많으면 **비용↑·속도↓** 입니다.

> 🚪 **다음 시간(2교시) 예고**: 위니브 문서가 5개, 더 길어지면
> 통째로 넣기는 힘듭니다. "**필요한 부분만 골라 넣는 법**"을
> 배웁니다 — 문서를 잘게 쪼개는 것부터 시작합니다.

---
# 🎓 배운 것 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | AI는 모르면 지어낸다(환각) | RAG = 문서 보고 답하기 |
| 2️⃣ 기본 | `llm.invoke()` 로 AI 부르기 | AI 호출의 기본 |
| 3️⃣ 응용 | 프롬프트로 역할·규칙 주기 | `prompt \| llm` |
| 4️⃣ 실습 | 문서 주니 거짓말→정확 | **이게 RAG!** |

**한 문장 요약**: AI에게 질문만 주면 지어내지만, **관련 문서를
같이 주면** 정확하게 답한다. 그게 RAG다.

**다음 시간**: 문서가 많고 길 때를 위해, 문서를 잘게
쪼개고 필요한 부분만 찾는 법을 배웁니다.

---
## 🤔 생각해보기 

위니브 문서가 5개 → 50개로 늘고, 각 문서도 10배 길어졌다고
상상해 보세요.

1. 셀 11에서 본 것처럼, 50개 문서를 매번 통째로 넣으면 어떤
   문제가 생길까요? (3가지 적어보기)
2. "통째로 넣기"의 가장 큰 약점을 한 문장으로 말한다면?
3. 50개 문서 전부 대신, AI에게 **무엇을** 주면 좋을까요?
   여러분의 아이디어를 자유롭게 적어보세요.

> ✍️ 메모:
> - 1번:
> - 2번:
> - 3번:
